# Teaching Qwen to hear Urdu 🇵🇰

**Qwen3-ASR** is Qwen's open speech-to-text model. It covers 30 languages, but **not Urdu**, a language of
230+ million speakers. In Pakistan, everyday communication runs on WhatsApp **voice notes**.

This notebook fine-tunes **Qwen3-ASR-0.6B** to transcribe Urdu, and measures every step:

1. [The problem](#1)
2. [Data: download and preparation](#2)
3. [Exploratory data analysis](#3)
4. [Baseline: the model before training](#4)
5. [Fine-tuning](#5)
6. [Results: before vs after](#6), plus [Roman Urdu output](#6b)
7. [Error analysis](#7)
8. [Summary and next steps](#8)

**Running it.** Select the project's `.venv` as the kernel and run all cells.
- Heavy steps call the project's scripts, which skip finished work. Re-running the notebook is cheap.
- The notebook kernel itself never uses the GPU. Only the evaluation and training scripts do, so pause the GPU
  services before sections 4–6.

In [ ]:
import json, os, random, re, subprocess, sys
from collections import Counter
from pathlib import Path

import jiwer
import librosa, librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, Markdown, Pretty, display
import warnings; warnings.filterwarnings("ignore", message="IProgress not found")

from common import for_scoring, read_jsonl, script_of

ROOT = Path.cwd()
DATA, RESULTS, RUN = ROOT / "data", ROOT / "results", ROOT / "runs" / "urdu-0.6b"
PY = sys.executable
RUN_TRAINING = True    # section 5 trains (2–4 hours) unless a finished run exists; False = only show results
EVAL_LIMIT = 0         # 0 = score every test clip; e.g. 1000 = a fixed random sample (faster)

# One colour per entity, used in every chart.
COLOR = {"common_voice": "#2a78d6", "fleurs": "#eb6834", "base": "#a3a29d", "base_forced": "#52514e",
         "finetuned": "#2a78d6"}
NAME = {"common_voice": "Common Voice", "fleurs": "FLEURS", "test_cv": "Common Voice test",
        "test_fleurs": "FLEURS test", "base": "Base, model picks the language",
        "base_forced": "Base, told it's Urdu", "finetuned": "Fine-tuned"}
plt.rcParams.update({"figure.figsize": (10, 3.6), "figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e"})
pd.set_option("display.max_colwidth", 140)


PROGRESS = re.compile(r"\d%(\||\s\()")  # tqdm bars ("12%|██") and the download bar ("12.0% (")


def sh(cmd):
    """Run a project script, streaming its output into the notebook; stop the notebook if it fails.
    Progress bars update a single line in place instead of printing thousands of lines."""
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    bar = None
    for line in p.stdout:
        if PROGRESS.search(line):
            bar = bar or display(Pretty(""), display_id=True)
            bar.update(Pretty(line.strip()))
        elif line.strip():
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")


def pct(x):
    return f"{100 * x:.1f}%"

<a id="1"></a>
## 1. The problem

These are the languages Qwen3-ASR officially supports:

In [ ]:
from qwen_asr.inference.utils import SUPPORTED_LANGUAGES
print(len(SUPPORTED_LANGUAGES), "languages:", ", ".join(SUPPORTED_LANGUAGES))
print("Urdu supported:", "Urdu" in SUPPORTED_LANGUAGES)

Hindi is on the list, and spoken Hindi and Urdu are almost the same language (Hindustani). The difference is the
**writing system**:
- Urdu is written in Perso-Arabic script, right to left: `آپ کیسے ہیں`.
- Hindi is written in Devanagari: `आप कैसे हैं`.

So our hypothesis is that the model already *hears* Urdu but writes it as Hindi. Section 4 measures this.

<a id="2"></a>
## 2. Data

| Dataset | What it is | Licence | Role here |
| --- | --- | --- | --- |
| [Common Voice 27.0, Urdu](https://mozilladatacollective.com/datasets/cmu5x1w3400cyo107st8cgsoq) (Mozilla) | Volunteers reading sentences on their own phones and laptops; ~6 GB (hours and speakers measured in section 3.1) | CC0 (no re-hosting of the audio) | Main training data + test set |
| [FLEURS, Urdu](https://huggingface.co/datasets/google/fleurs) (Google) | Wikipedia sentences read by native speakers; ~10 hours | CC-BY-4.0 | Extra training data + an *independent* test set |

**Common Voice needs a free account.**
1. Accept the dataset's terms on its page.
2. Put `MDC_API_KEY=<your key>` (from Account → Credentials) in a file named `.env` next to this notebook.
   `.gitignore` keeps that file out of the repo.

Without the key, the notebook continues with FLEURS only.

In [ ]:
cv_ready = any((DATA / "raw" / "common_voice").glob("**/validated.tsv"))
has_key = bool(os.environ.get("MDC_API_KEY")) or (ROOT / ".env").exists()
if not cv_ready and not has_key:
    print("No Mozilla API key found: downloading FLEURS only. See the instructions above to add Common Voice.")
sh(f"{PY} download.py" + (" --skip-cv" if cv_ready or not has_key else ""))

### Preparation

`prepare.py` decodes every clip once to 16 kHz mono WAV and writes four lists:

| File | Contents |
| --- | --- |
| `train.jsonl` | Common Voice validated clips + FLEURS train |
| `dev.jsonl` | 500 clips watched during training (never trained on) |
| `test_cv.jsonl` | Common Voice's official test split |
| `test_fleurs.jsonl` | FLEURS test split |

**Making the test fair.** Training excludes every *speaker* who appears in Common Voice dev/test, and every
*sentence* that appears in any dev/test set. Section 3.8 verifies this.

In [ ]:
sh(f"{PY} prepare.py")

In [ ]:
SPLITS = [s for s in ("train", "dev", "test_cv", "test_fleurs") if (DATA / f"{s}.jsonl").exists()]
df = pd.concat([pd.DataFrame(read_jsonl(DATA / f"{s}.jsonl")).assign(split=s) for s in SPLITS], ignore_index=True)
if "speaker" not in df:
    df["speaker"] = None
df["norm"] = df.ref.map(for_scoring)                       # the text exactly as it is scored
df["words"] = df.norm.str.split().str.len()
df["chars_per_sec"] = df.norm.str.replace(" ", "").str.len() / df.seconds
df.head(3)[["id", "split", "source", "seconds", "ref"]]

<a id="3"></a>
## 3. Exploratory data analysis

### 3.1 Overview

In [ ]:
overview = (df.groupby(["split", "source"], sort=False)
              .agg(clips=("id", "size"), hours=("seconds", lambda s: s.sum() / 3600),
                   speakers=("speaker", "nunique"), median_sec=("seconds", "median"),
                   median_words=("words", "median"))
              .round(2))
overview.loc[overview.index.get_level_values("source") == "fleurs", "speakers"] = np.nan  # FLEURS has no speaker IDs
display(overview)
train = df[df.split == "train"]
print(f"Training audio: {train.seconds.sum() / 3600:.1f} hours in {len(train):,} clips")

### 3.2 Clip durations

Duration matters for three reasons:
- **Memory:** a batch is padded to its longest clip.
- **Filtering:** `prepare.py` drops clips outside 0.5–30 s, plus clips whose label can't match the audio: romanized text (a few Common Voice sentences are typed in Latin letters) or speech faster than 20 characters per second (cut-off audio; see 3.5).
- **Realism:** voice notes are often longer than read sentences.

In [ ]:
sources = [s for s in ("common_voice", "fleurs") if (df.source == s).any()]
fig, axes = plt.subplots(1, len(sources), sharex=True, squeeze=False)
bins = np.arange(0, 31, 1)
for ax, s in zip(axes[0], sources):
    sec = df[df.source == s].seconds
    ax.hist(sec, bins=bins, color=COLOR[s], rwidth=0.85)
    ax.axvline(sec.median(), color="#0b0b0b", lw=1, ls="--")
    ax.text(sec.median(), ax.get_ylim()[1] * 0.92, f"  median {sec.median():.1f}s", fontsize=9)
    ax.set(title=NAME[s], xlabel="seconds per clip", ylabel="clips")
plt.tight_layout(); plt.show()

### 3.3 Speakers (Common Voice)

Crowd-sourced data is usually dominated by a few very active volunteers. If a handful of voices make up most of
the training audio, the model can overfit to them. That's why the test set uses *unseen* speakers.

In [ ]:
cv_train = train[train.source == "common_voice"]
if len(cv_train):
    per_speaker = cv_train.speaker.value_counts()
    share = per_speaker.cumsum() / per_speaker.sum()
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.plot(np.arange(1, len(share) + 1), 100 * share.values, color=COLOR["common_voice"], lw=2)
    ax.set(title="How much of the training audio comes from the most active speakers",
           xlabel="speakers, most active first", ylabel="% of training clips", ylim=(0, 101), xscale="log")
    plt.tight_layout(); plt.show()
    print(f"{len(per_speaker)} training speakers. Top 1 speaker: {pct(share.iloc[0])} of clips; "
          f"top 10: {pct(share.iloc[min(9, len(share) - 1)])}; median speaker: {per_speaker.median():.0f} clips.")
else:
    print("Common Voice not prepared yet: skipped.")

### 3.4 Who is speaking? (Common Voice demographics)

These are self-reported, optional fields, so many speakers are "unknown". A strong skew (e.g. mostly young male
voices) is a known cause of higher error rates for everyone else. It's worth stating in the write-up.

In [ ]:
tsv = sorted((DATA / "raw" / "common_voice").glob("**/validated.tsv"))
if tsv:
    meta = pd.read_csv(tsv[-1], sep="\t", quoting=3, dtype=str, keep_default_na=False)
    meta = meta[meta.path.str.rsplit(".", n=1).str[0].isin(cv_train.id.str.split("/").str[1])]
    cols = [c for c in ("gender", "age") if c in meta]
    for c in cols:
        counts = meta[c].replace("", "unknown").value_counts()
        display(pd.DataFrame({"clips": counts, "share": (counts / counts.sum()).map(pct)}).rename_axis(c))
    if not cols:
        print("This Common Voice release has no gender/age columns.")
else:
    print("Common Voice not downloaded yet: skipped.")

### 3.5 Sentences and speaking rate

**Speaking rate** (characters per second) is a cheap data-quality check:
- a very *slow* clip is usually mostly silence or a truncated reading;
- a very *fast* one often doesn't match its transcript.

The extremes are listed below, so you can listen to a few.

In [ ]:
fig, axes = plt.subplots(1, 2)
for s in sources:
    d = df[df.source == s]
    axes[0].hist(d.words, bins=np.arange(0, 60, 2), color=COLOR[s], alpha=0.75, rwidth=0.85, label=NAME[s], density=True)
    axes[1].hist(d.chars_per_sec.clip(upper=25), bins=np.arange(0, 26, 1), color=COLOR[s], alpha=0.75,
                 rwidth=0.85, label=NAME[s], density=True)
axes[0].set(title="Words per sentence", xlabel="words", ylabel="share of clips")
axes[1].set(title="Speaking rate", xlabel="characters per second", ylabel="share of clips")
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

lo, hi = train.chars_per_sec.quantile([0.005, 0.995])
odd = train[(train.chars_per_sec < lo) | (train.chars_per_sec > hi)]
print(f"{len(odd)} training clips ({pct(len(odd) / len(train))}) are outside the 0.5–99.5% speaking-rate range "
      f"({lo:.1f}–{hi:.1f} chars/s). Most extreme:")
display(odd.reindex(odd.chars_per_sec.sub(train.chars_per_sec.median()).abs().sort_values(ascending=False).index)
           .head(6)[["id", "seconds", "chars_per_sec", "ref"]].round(1))

### 3.6 Vocabulary

**Out-of-vocabulary (OOV)** words are test words that never appear in the training transcripts. The model can
still spell them out from sounds, but they're where most errors happen. A high OOV rate on FLEURS
(Wikipedia sentences: names, places, technical words) means FLEURS is the harder, more honest test.

In [ ]:
vocab = Counter(w for t in train.norm for w in t.split())
rows = []
for s in [s for s in SPLITS if s.startswith("test")]:
    words = [w for t in df[df.split == s].norm for w in t.split()]
    rows.append({"test set": NAME[s], "words": len(words),
                 "OOV rate (word tokens unseen in training)": pct(sum(w not in vocab for w in words) / max(len(words), 1))})
print(f"Training vocabulary: {len(vocab):,} distinct words, {sum(vocab.values()):,} word tokens.")
display(pd.DataFrame(rows))
display(pd.DataFrame(vocab.most_common(15), columns=["most frequent words", "count"]).T)

### 3.7 Script mix: English words and digits

Real Urdu speech is full of English words: "meeting", "computer". Here we check how often the *transcripts*
contain Latin-script words or digits, i.e. what the model is being taught to write for them.

In [ ]:
has_latin = df.ref.str.contains(r"[A-Za-z]")
has_digit = df.ref.str.contains(r"[0-9٠-٩۰-۹]")
display(pd.DataFrame({"sentences with Latin letters": has_latin.groupby(df.source).mean().map(pct),
                      "sentences with digits": has_digit.groupby(df.source).mean().map(pct)}).rename(index=NAME))
display(df[has_latin].head(3)[["source", "ref"]])

### 3.8 Leakage checks

The test sets must contain no training speakers and no training sentences. Otherwise the "after" number would
be inflated.

In [ ]:
train_spk = set(train.speaker.dropna())
train_txt = set(train.norm)
for s in [s for s in SPLITS if s != "train"]:
    d = df[df.split == s]
    spk = len(set(d.speaker.dropna()) & train_spk)
    txt = int(d.norm.isin(train_txt).sum())
    print(f"{NAME.get(s, s):>18}: {spk} shared speakers, {txt} shared sentences  {'OK' if spk == txt == 0 else 'LEAK'}")

### 3.9 Listen and look

A **mel spectrogram** is how the model "sees" audio:
- time runs left to right;
- pitch (frequency) goes up the y-axis, on a scale that matches human hearing;
- brightness is loudness.

Qwen3-ASR's audio encoder turns pictures like these into tokens for the language model.

In [ ]:
for _, r in train.sample(2, random_state=0).iterrows():
    y, sr = librosa.load(r.audio, sr=16000)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), ref=np.max)
    fig, ax = plt.subplots(figsize=(10, 2.6))
    librosa.display.specshow(mel, sr=sr, x_axis="time", y_axis="mel", ax=ax, cmap="magma")
    ax.set(title=f"{NAME[r.source]} · {r.seconds:.1f}s", xlabel="seconds", ylabel="Hz")
    plt.tight_layout(); plt.show()
    display(Markdown(f"> {r.ref}"), Audio(y, rate=sr))

<a id="4"></a>
## 4. Baseline: the model before training

`evaluate.py` transcribes every test clip with greedy decoding (no randomness) and reports:
- **WER** (word error rate): substitutions + deletions + insertions, divided by the number of reference words.
  0 is perfect. It can exceed 100% when the model inserts extra words.
- **CER** (character error rate): the same per character. It is kinder to Urdu's spelling variants.

Before scoring, both texts are normalized: no punctuation or diacritics, consistent letters, Western digits.

⚠️ Pause the GPU services before running this.

In [ ]:
def evaluate(name, model=None, language=None, sets=None):
    """Run evaluate.py unless results for this exact model already exist; return the results.
    language forces the output script (needed once a model knows both Urdu and Roman Urdu)."""
    path = RESULTS / f"{name}.json"
    want = str(model or "Qwen/Qwen3-ASR-0.6B")
    if not path.exists() or json.loads(path.read_text())["model"] != want:
        sh(f"{PY} evaluate.py --name {name} --model {want}" + (f" --limit {EVAL_LIMIT}" if EVAL_LIMIT else "")
           + (f' --language "{language}"' if language else "") + (f" --sets {sets}" if sets else ""))
    return json.loads(path.read_text())


def predictions(name):
    return pd.concat([pd.DataFrame(read_jsonl(f)).assign(set=f.stem.split("_", 1)[1])
                      for f in sorted(RESULTS.glob(f"{name}_test_*.jsonl"))], ignore_index=True)


base = evaluate("base")
pd.DataFrame({NAME[s]: {"WER": pct(v["wer"]), "CER": pct(v["cer"]),
                        "written in": ", ".join(f"{k} {pct(n / v['clips'])}" for k, n in v["output_script"].items()),
                        "language the model detected": ", ".join(f"{k} {pct(n / v['clips'])}" for k, n in v["detected_language"].items())}
              for s, v in base["sets"].items()})

The hypothesis holds: the base model labels the clips as **Hindi** and writes them in **Devanagari**. Below,
the model's output next to the reference. You'll often see that the *words are right* and only the script is
wrong:

In [ ]:
base_pred = predictions("base")
base_pred.sample(min(6, len(base_pred)), random_state=1)[["set", "ref", "pred", "language"]]

### A stronger baseline: tell the model it's Urdu

Qwen3-ASR accepts a forced language: the prompt then ends with `language Urdu<asr_text>` and the model only writes
the text. Urdu isn't on its list, but the decoder is a multilingual LLM that has seen Urdu text, so it can write
Urdu script when told to. This is the fairest "before": the fine-tune has to beat *this*, not only the Hindi
output.

In [ ]:
base_forced = evaluate("base_forced", language="Urdu")
pd.DataFrame({NAME[s]: {"WER": pct(v["wer"]), "CER": pct(v["cer"]),
                        "written in": ", ".join(f"{k} {pct(n / v['clips'])}" for k, n in v["output_script"].items())}
              for s, v in base_forced["sets"].items()})

In [ ]:
side = predictions("base_forced")[["id", "set", "ref", "pred"]]
side.sample(min(6, len(side)), random_state=1)

<a id="5"></a>
## 5. Fine-tuning

**Method: full fine-tune with Qwen's official script** (`qwen3_asr_sft.py`, plus one fix: it scaled the loss and gradients up by the gradient-accumulation factor, 8×; see the line marked FIX).
- All 938M weights are updated: audio encoder + language model.
- Community reports found LoRA adapters worked worse for Qwen3-ASR.
- Weights are kept in bf16, so the model, its gradients and the optimizer state fit in about 8 GB of a 16 GB GPU.

**Training target.** Each example teaches the model to answer `language Urdu<asr_text>` + the transcript. The
model learns to name the language itself, the same format it uses for its 30 built-in languages.

| Setting | Value | Why |
| --- | --- | --- |
| Batch | 2 clips × 32 accumulation = 64 clips per update | 4+ clips run out of memory on the longest (30 s) clips; 64 per update keeps updates stable |
| Learning rate | 2e-5, linear decay, 2% warm-up | Qwen's default for this script |
| Epochs | 2 | Data is modest (~43 h, 79 Common Voice speakers); more risks memorizing speakers |
| Dev loss | every 200 updates | Early warning for over-fitting |

**Running it.** With `RUN_TRAINING = True` the next cell runs `./train.sh` (2–4 hours). The GPU services must stay
paused, and VS Code must stay open: closing it stops the kernel and the training. If anything interrupts it, run the
cell again; it resumes from the newest checkpoint (saved every 200 updates). Once a run has finished, the cell skips
training, so *Run All* later only rebuilds the charts. Prefer a terminal? `./train.sh` there does the same thing,
and this notebook picks up its checkpoints.

In [ ]:
def checkpoints():
    return sorted(RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))


def finished():
    """True once the newest checkpoint is the last step of the run."""
    if not checkpoints():
        return False
    state = json.loads((checkpoints()[-1] / "trainer_state.json").read_text())
    return state["global_step"] >= state["max_steps"]


if RUN_TRAINING and not finished():
    sh("./train.sh")
elif finished():
    print("Training already finished; skipping (delete runs/urdu-0.6b to train again).")

ckpts = checkpoints()
print(f"{len(ckpts)} checkpoint(s) in {RUN}:", [p.name for p in ckpts] or "none yet: train first")

### Training curves

- **Training loss** is the error on the batches being learned.
- **Dev loss** is the error on clips the model never trains on.

When dev loss stops falling while training loss keeps going down, the model has started memorizing. The best
checkpoint is the one with the lowest dev loss.

In [ ]:
BEST = None
if ckpts:
    history = json.loads((ckpts[-1] / "trainer_state.json").read_text())["log_history"]
    tr = pd.DataFrame([h for h in history if "loss" in h])
    ev = pd.DataFrame([h for h in history if "eval_loss" in h])
    fig, ax = plt.subplots()
    if len(tr):
        ax.plot(tr.step, tr.loss, color=COLOR["finetuned"], lw=1.5, alpha=0.6, label="training loss")
    if len(ev):
        ax.plot(ev.step, ev.eval_loss, color=COLOR["fleurs"], lw=2, marker="o", ms=8, label="dev loss")
        best_step = int(ev.loc[ev.eval_loss.idxmin(), "step"])
        ax.axvline(best_step, color="#52514e", ls="--", lw=1)
        ax.text(best_step, ax.get_ylim()[1] * 0.95, f"lowest dev loss (step {best_step})  ", fontsize=9, ha="right")
        saved = {int(p.name.split("-")[1]): p for p in ckpts}
        BEST = saved.get(best_step)
    ax.set(title="Training progress", xlabel="update step", ylabel="loss")
    ax.legend(frameon=False)
    plt.tight_layout(); plt.show()
    BEST = BEST or ckpts[-1]  # the best one may have been rotated out; then use the newest
    print("Checkpoint used for evaluation:", BEST.name)

<a id="6"></a>
## 6. Results: before vs after

Same test clips, same scoring code, same greedy decoding. Two "before" numbers: the base model choosing the
language itself (it picks Hindi), and the base model told the language is Urdu. The fine-tuned model is told
`Urdu` too, since it also knows Roman Urdu.

In [ ]:
ft = evaluate("finetuned", BEST, language="Urdu") if BEST else None
if ft:
    sets = [s for s in base["sets"] if s in ft["sets"]]
    fig, axes = plt.subplots(1, 2, sharey=True)
    x = np.arange(len(sets))
    for ax, metric in zip(axes, ("wer", "cer")):
        for i, (name, res) in enumerate((("base", base), ("base_forced", base_forced), ("finetuned", ft))):
            vals = [100 * res["sets"][s][metric] for s in sets]
            bars = ax.bar(x + (i - 1) * 0.28, vals, width=0.26, color=COLOR[name], label=NAME[name])
            ax.bar_label(bars, fmt="%.0f%%", padding=2, fontsize=8, color="#0b0b0b")
        ax.set(title=f"{metric.upper()} (lower is better)", xticks=x, xticklabels=[NAME[s] for s in sets])
        ax.set_ylabel("error rate (%)")
    top_val = max(100 * r["sets"][s][m] for r in (base, base_forced, ft) for s in sets for m in ("wer", "cer"))
    axes[0].set_ylim(0, top_val * 1.15)
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=3, frameon=False)
    plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()

    table = pd.DataFrame([{"test set": NAME[s], "clips": ft["sets"][s]["clips"],
                           "WER base (auto)": pct(base["sets"][s]["wer"]),
                           "WER base (told Urdu)": pct(base_forced["sets"][s]["wer"]),
                           "WER fine-tuned": pct(ft["sets"][s]["wer"]),
                           "CER base (auto)": pct(base["sets"][s]["cer"]),
                           "CER base (told Urdu)": pct(base_forced["sets"][s]["cer"]),
                           "CER fine-tuned": pct(ft["sets"][s]["cer"])} for s in sets])
    display(table)
    print("Markdown for the README:\n")
    print("| " + " | ".join(table.columns) + " |\n|" + " --- |" * len(table.columns))
    for row in table.itertuples(index=False):
        print("| " + " | ".join(map(str, row)) + " |")
else:
    print("No fine-tuned checkpoint yet: train first (section 5).")

In [ ]:
if ft:
    side = predictions("base")[["id", "set", "ref", "pred"]].merge(
        predictions("finetuned")[["id", "pred"]], on="id", suffixes=(" (before)", " (after)"))
    display(side.sample(min(8, len(side)), random_state=2).drop(columns="id"))

<a id="6b"></a>
### 6b. Roman Urdu output (stage 2)

Most Pakistanis *type* Urdu in Latin letters. No speech dataset has Roman Urdu transcripts, so they are made:
1. **A text model learns to romanize.** Qwen3.5-0.8B is LoRA-fine-tuned on Roman-Urdu-Parl sentence pairs
   (`romanize/train.py`).
2. **It is measured on human data.** Dakshina has 4,880 Urdu sentences romanized by native speakers, never trained
   on. Published reference points on this test (char-BLEU): GPT-4o mini 75.1, m2m100 78.5.
3. **It romanizes our human Urdu transcripts**, and the speech model trains on both versions of every clip. The
   tag `language Urdu` or `language Roman urdu` picks which script it writes.

Roman Urdu has no standard spelling ("nahi" / "nahin"), so no model matches a single reference perfectly. The
speech model's Roman Urdu scores below are measured against the romanizer's output for the *human* test
transcripts, not against human romanizations.

In [ ]:
ROM = ROOT / "romanize" / "results"
rom_res = {n: json.loads((ROM / f"{n}.json").read_text()) for n in ("base", "finetuned") if (ROM / f"{n}.json").exists()}
if rom_res:
    display(pd.DataFrame({("Qwen3.5-0.8B, untouched" if n == "base" else "Qwen3.5-0.8B, fine-tuned"):
                          {"CER (lower better)": pct(r["cer"]), "WER (lower better)": pct(r["wer"]),
                           "char-BLEU (higher better)": r["char_bleu"], "chrF (higher better)": r["chrf"]}
                          for n, r in rom_res.items()}))
    if (ROM / "finetuned_test_dakshina.jsonl").exists():
        ex = pd.DataFrame(read_jsonl(ROM / "finetuned_test_dakshina.jsonl"))
        display(ex.sample(min(6, len(ex)), random_state=0).rename(columns={"ref": "human romanization",
                                                                           "pred": "fine-tuned output"}))
else:
    print("Romanizer not evaluated yet: see README, stage 2.")

ft_rom = None
if BEST and (DATA / "test_cv_roman.jsonl").exists():
    ft_rom = evaluate("finetuned_roman", BEST, language="Roman urdu", sets="test_cv_roman test_fleurs_roman")
    display(pd.DataFrame({NAME[s.removesuffix("_roman")]: {"WER": pct(v["wer"]), "CER": pct(v["cer"])}
                          for s, v in ft_rom["sets"].items()}))
    both = predictions("finetuned")[["id", "pred"]].merge(
        predictions("finetuned_roman").assign(id=lambda d: d.id.str.removesuffix("#roman"))[["id", "pred"]],
        on="id", suffixes=(" (Urdu)", " (Roman Urdu)"))
    display(both.sample(min(6, len(both)), random_state=3).drop(columns="id"))

<a id="7"></a>
## 7. Error analysis (fine-tuned model)

Where does the model still go wrong? Three views:
1. **Per-clip WER:** how many clips are perfect, and how bad is the tail.
2. **WER by clip length:** short clips have few words, so a single error weighs a lot.
3. **Most frequent word substitutions:** many turn out to be spelling variants (`گئے`/`گئی`, `کیلئے`/`کے لیے`)
   rather than real mishearings. That's worth stating honestly in the write-up.

In [ ]:
if ft:
    p = predictions("finetuned")
    p["ref_n"], p["pred_n"] = p.ref.map(for_scoring), p.pred.map(for_scoring)
    p["wer"] = [jiwer.wer(r, h) if r else 0.0 for r, h in zip(p.ref_n, p.pred_n)]
    seconds = df.drop_duplicates("id").set_index("id").seconds
    p["seconds"] = p.id.map(seconds)

    fig, axes = plt.subplots(1, 2)
    axes[0].hist(100 * p.wer.clip(upper=1), bins=np.arange(0, 105, 5), color=COLOR["finetuned"], rwidth=0.85)
    axes[0].set(title="Per-clip WER", xlabel="WER of the clip (%)", ylabel="clips")
    buckets = pd.cut(p.seconds, [0, 3, 5, 8, 12, 20, 31])
    by_len = p.groupby(buckets, observed=True).apply(lambda d: jiwer.wer(list(d.ref_n), list(d.pred_n)))
    axes[1].bar([str(b) for b in by_len.index], 100 * by_len.values, color=COLOR["finetuned"], width=0.7)
    axes[1].set(title="WER by clip length", xlabel="clip length (seconds)", ylabel="WER (%)")
    plt.tight_layout(); plt.show()
    print(f"Perfect transcripts: {pct((p.wer == 0).mean())} of clips; "
          f"clips with WER above 50%: {pct((p.wer > 0.5).mean())}")

    out = jiwer.process_words(list(p.ref_n), list(p.pred_n))
    subs, ins, dels = Counter(), Counter(), Counter()   # word-level (jiwer groups neighbouring errors into spans)
    for ref, hyp, chunks in zip(out.references, out.hypotheses, out.alignments):
        for c in chunks:
            r, h = ref[c.ref_start_idx:c.ref_end_idx], hyp[c.hyp_start_idx:c.hyp_end_idx]
            if c.type == "substitute":
                subs.update(zip(r, h))
            elif c.type == "delete":
                dels.update(r)
            elif c.type == "insert":
                ins.update(h)
    display(pd.DataFrame([(r, h, n) for (r, h), n in subs.most_common(15)],
                         columns=["reference word", "model wrote", "times"]))
    top = lambda c: ([w for w, _ in c.most_common(8)] + [""] * 8)[:8]
    display(pd.DataFrame({"most deleted words": top(dels), "most inserted words": top(ins)}).T)
    display(Markdown("**Worst clips** (listen to a couple: is it the model, or the recording?)"))
    display(p.sort_values("wer", ascending=False).head(5)[["id", "seconds", "wer", "ref", "pred"]].round(2))

<a id="8"></a>
## 8. Summary and next steps

**What was done**
- Measured that Qwen3-ASR-0.6B hears Urdu but, left to itself, writes it as Hindi (WER near 100%). Told the
  language is Urdu, it writes Urdu script but misspells heavily (section 4): that is the baseline to beat.
- Fully fine-tuned it on ~43 hours of openly licensed Urdu speech, with speaker- and sentence-disjoint test sets.
- Before/after numbers are in section 6.

**Limitations to state honestly**
- Training and test audio is *read* speech. Real voice notes are spontaneous, noisy, and mix in English more
  often.
- The Common Voice speaker and demographic skew from sections 3.3–3.4 carries into the model.
- WER penalizes valid spelling variants. CER is the fairer headline number for Urdu.

**Next steps**
1. **Roman Urdu with human references:** have native speakers romanize a few hundred test transcripts, to score
   the Roman Urdu output against people rather than the romanizer (section 6b).
2. **Quantize and run fully offline** (GGUF / llama.cpp) for the local demo.
3. **Test on real-style voice notes**, recorded with consent, to measure the gap between read and spontaneous
   speech.